# Applied-paper figures (v2)

One notebook that regenerates **every** figure for the applied energy-ABM paper and
saves each as a PNG (`dpi=200`, `bbox_inches="tight"`) into the tracked directory
`research/applied/figures/`.

Sources are the committed calibration (`household_energy/calibrated_config.yaml`),
the re-locked rollups (`results_lsoa/transfer_v2_*`), the SERL ledger helpers
(`fit_serl_ledger`), and DESNZ (`utils.load_desnz`). Figures 3 and 8 run small
Newcastle model / segmentation samples; the rest read committed caches.

Run end-to-end with:
```
./.venv/bin/jupyter nbconvert --to notebook --execute \
  research/applied/notebooks/4_figures_v2.ipynb \
  --output-dir /tmp --output fig_check.ipynb --ExecutePreprocessor.timeout=1200
```

## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
%matplotlib inline

# Discover the repo root by walking up from the working directory.
REPO = Path.cwd().resolve()
while not (REPO / "household_energy").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "household_energy").is_dir(), f"could not locate repo root from {Path.cwd()}"

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "research/applied/scripts"))
import fit_serl_ledger as L          # SERL ledger helpers
from utils import load_desnz         # DESNZ loader

FIGDIR = REPO / "research/applied/figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

CONFIG = REPO / "household_energy" / "calibrated_config.yaml"

# Shared SERL reads used by several figures.
daily = pd.read_csv(L.DAILY)
diurnal = pd.read_csv(L.DIURNAL)

# The eight-filter SERL cell accessor (identical to L.cells; copied from the
# calibration/validation notebooks so this notebook stands alone).
def cell(quantity, hf, seg3_var="none", seg3_value="none", period_type="monthly"):
    return daily[(daily.quantity == quantity) & (daily.heating_fuel == hf) &
                 (daily.seg3_var == seg3_var) & (daily.seg3_value.astype(str) == str(seg3_value)) &
                 (daily.period_type == period_type) & (daily.year == L.YEAR) &
                 (daily.weekday_weekend == L.WKND) & (daily.has_pv == L.PV)]

print(f"REPO   = {REPO}")
print(f"FIGDIR = {FIGDIR}")
print(f"SERL year {L.YEAR}, weekday_weekend={L.WKND}, has_pv={L.PV}")

## Shared Newcastle segmentation run

One `run_pooled_segmentations` call on a 1,500-home Newcastle sample, reused by
Figure 8. Figure 3 runs its own `EnergyModel` sample (kept as in the source
script, but it reads a committed profile cache when present).

In [ ]:
import yaml
from household_energy.serl_calibration_v2 import run_pooled_segmentations

vcfg = yaml.safe_load(Path(CONFIG).read_text())["model"]
ovr = {f"model.{k}": v for k, v in vcfg.items()}
seg = run_pooled_segmentations(
    repo_root=REPO, target_year=L.YEAR,
    seg3_vars=["central_heating_type", "floor_area_m2", "currentEnergyRating",
               "building_type", "num_occupants"],
    cities=["newcastle"], max_homes_per_city=1500, seed=7,
    model_overrides=ovr, process_column_preferred="lsoa_code", n_procs=4)
print(f"ran the ABM on Newcastle for {L.YEAR}: {len(seg):,} rows of hourly output")

## Figure 1 — `figure_calibration_serl_match.png`
Monthly seasonal demand, model vs SERL, per home per day, both fuels
(ported from `make_monthly_figure.py`).

In [ ]:
MODEL = REPO / "results/serl_ledger/abm_reference_newcastle.csv"
SERL = REPO / "results_lsoa/serl_monthly_reference_2023.csv"

SERL_GREY, MODEL_C = "#8a8a8a", "#2E8B57"
DAYS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
MONTHS = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

m = pd.read_csv(MODEL)
s = pd.read_csv(SERL).sort_values("month")

series = {
    "Gas (gas-heated dwellings)": (
        m[m.cohort == "gas"].sort_values("month").gas.to_numpy(),
        s.gas_gas.to_numpy() / DAYS),
    "Electricity (electric-heated dwellings)": (
        m[m.cohort == "elec"].sort_values("month").elec.to_numpy(),
        s.electric_elec.to_numpy() / DAYS),
}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9), constrained_layout=True)
x = np.arange(1, 13)
for ax, (title, (model, serl)) in zip(axes, series.items()):
    ax.plot(x, serl, color=SERL_GREY, lw=1.6, ls="--", marker="o", ms=4, label="SERL")
    ax.plot(x, model, color=MODEL_C, lw=2.2, marker="o", ms=4, label="model")
    ax.set_title(title, fontsize=10.5)
    ax.set_xlabel("month")
    ax.set_xticks(x); ax.set_xticklabels(MONTHS)
    ax.set_ylim(bottom=0)
    ax.grid(True, color="#eeeeee", zorder=0)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.legend(fontsize=8.5, frameon=False, loc="upper right")
axes[0].set_ylabel("demand (kWh / home / day)")

out = FIGDIR / "figure_calibration_serl_match.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"wrote {out}")

## Figure 2 — `figure_temp_response.png`
Daily demand vs outdoor temperature with the SERL-fitted heating hinge
(ported from `make_temp_response_figure.py`).

In [ ]:
from scipy.optimize import least_squares

def temp_band_rows(quantity, hf):
    s = daily[(daily.quantity == quantity) & (daily.heating_fuel == hf) &
              (daily.seg3_var == "temperature_band") & (daily.period_type == "annual") &
              (daily.year == L.YEAR) & (daily.weekday_weekend == L.WKND) & (daily.has_pv == L.PV)]
    return s.dropna(subset=["mean", "mean_temp"]).sort_values("mean_temp")

def hinge_fit(quantity, hf):
    s = temp_band_rows(quantity, hf)
    T = s["mean_temp"].to_numpy(float); y = s["mean"].to_numpy(float)
    w = np.sqrt(pd.to_numeric(s["n_rounded"], errors="coerce").fillna(1).to_numpy(float))
    fit = least_squares(lambda p: w * (p[0] + p[1] * np.maximum(0.0, p[2] - T) - y),
                        x0=[y.min(), 1.0, 15.0], bounds=([0, 0, 8], [50, 30, 22]))
    return dict(baseline=float(fit.x[0]), slope_per_day=float(fit.x[1]), setpoint=float(fit.x[2]))

def panel(ax, quantity, hf, colour, title):
    f = hinge_fit(quantity, hf)
    ref = L.joint_setpoint_slope(quantity, hf)
    assert all(np.isclose(f[k], ref[k]) for k in ("baseline", "slope_per_day", "setpoint")), \
        f"drifted from pipeline: {hf}"
    s = temp_band_rows(quantity, hf)
    T, y = s["mean_temp"].to_numpy(float), s["mean"].to_numpy(float)
    xx = np.linspace(T.min(), T.max(), 60)
    yy = f["baseline"] + f["slope_per_day"] * np.maximum(0.0, f["setpoint"] - xx)
    ax.scatter(T, y, color=colour, zorder=3, s=52, label="SERL measured")
    ax.plot(xx, yy, color="#2E8B57", lw=2.4,
            label=f"model response (setpoint {f['setpoint']:.1f}\u00b0C)")
    ax.axhline(f["baseline"], color=GREY, ls=":", lw=1.1,
               label=f"weather-independent baseline {f['baseline']:.1f} kWh/day")
    ax.axvline(f["setpoint"], color=colour, ls="--", lw=0.8, alpha=0.4, zorder=1)
    ax.set_xlabel("outdoor temperature (\u00b0C)")
    ax.set_ylabel(f"{title.lower()} used (kWh/day)")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8, frameon=False, loc="upper right")
    ax.grid(True, color="#eeeeee", zorder=0)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    return f

GREY = "#8a8a8a"
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9), constrained_layout=True)
gj = panel(axes[0], "Gas", "Gas", "#c0392b", "Gas (current heating)")
ej = panel(axes[1], "Electricity imports", "Electric", "#2c5f9e", "Electricity (electric-heated)")

out = FIGDIR / "figure_temp_response.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"wrote {out}")
print(f"gas : setpoint {gj['setpoint']:.2f}\u00b0C, slope {gj['slope_per_day']/24:.4f} kWh/h/\u00b0C, "
      f"baseline {gj['baseline']:.2f} kWh/day")
print(f"elec: setpoint {ej['setpoint']:.2f}\u00b0C, slope {ej['slope_per_day']/24:.4f} kWh/h/\u00b0C, "
      f"baseline {ej['baseline']:.2f} kWh/day")

## Figure 3 — `figure_intraday_shape.png`
Within-day winter/summer shape, model vs SERL 10-90% envelope
(ported from `make_intraday_figure.py`). Runs `EnergyModel` on a 20-LSOA
Newcastle sample only when the committed profile cache is absent.

In [ ]:
from household_energy.model import EnergyModel

YEAR = 2023
SAMPLE_LSOAS = 20
GEO = REPO / "data/epc_abm_newcastle.geojson"
CLIM = REPO / "data/ncc_2t_timeseries_2010_2026.parquet"
ENVELOPE = REPO / "data/serl_profiles/serl_profiles_num_occupants.csv"

SERL_GREY, WINTER_C, SUMMER_C = "#8a8a8a", "#2E8B57", "#E1A100"
PROF_CACHE = REPO / "research/applied/results/serl_ledger/intraday_profiles_newcastle.csv"

def compute_profiles():
    gdf = gpd.read_file(GEO); gdf["lsoa_code"] = gdf["lsoa_code"].astype(str)
    _all = sorted(gdf.lsoa_code.unique())
    SAMPLE = _all[:: max(1, len(_all) // SAMPLE_LSOAS)][:SAMPLE_LSOAS]
    gdf = gdf[gdf.lsoa_code.isin(SAMPLE)].copy()
    mdl = EnergyModel(gdf=gdf, climate_parquet=str(CLIM),
                      climate_start=pd.Timestamp(f"{YEAR}-01-01", tz="UTC"),
                      local_tz="Europe/London", collect_agent_level=False, config_path=str(CONFIG))
    ag = mdl.household_agents
    print(f"running {len(ag)} dwellings across {len(SAMPLE)} LSOAs for {YEAR} (8760 hours)...")
    hrE = np.zeros(8760); hrG = np.zeros(8760)
    for t in range(8760):
        mdl.step()
        e = g = 0.0
        for a in ag:
            e += getattr(a, "electric_kwh", 0.0); g += getattr(a, "gas_kwh", 0.0)
        hrE[t] = e; hrG[t] = g
    ts = pd.date_range(f"{YEAR}-01-01", periods=8760, freq="h", tz="UTC").tz_convert("Europe/London")
    dfh = pd.DataFrame({"e": hrE, "g": hrG, "hour": ts.hour, "month": ts.month})
    rows = []
    for fc in ("e", "g"):
        for name, months in (("winter", [12, 1, 2]), ("summer", [6, 7, 8])):
            p = dfh[dfh.month.isin(months)].groupby("hour")[fc].mean()
            p = p / p.mean()
            for hour, val in p.items():
                rows.append({"fuel": fc, "season": name, "hour": hour, "value": val})
    out_df = pd.DataFrame(rows)
    PROF_CACHE.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(PROF_CACHE, index=False)
    print(f"cached profiles -> {PROF_CACHE}")
    return out_df

cache = pd.read_csv(PROF_CACHE) if PROF_CACHE.exists() else compute_profiles()
PROF = {fc: {ss: g.set_index("hour")["value"].sort_index()
             for ss, g in cache[cache.fuel == fc].groupby("season")}
        for fc in ("e", "g")}

serl_prof = pd.read_csv(ENVELOPE)

def serl_envelope(fuel):
    s = serl_prof[(serl_prof["kind"] == "hourly") & (serl_prof["fuel"] == fuel)]
    gq = s.groupby("idx")["mult"]
    return pd.DataFrame({"lo": gq.quantile(0.10), "hi": gq.quantile(0.90),
                         "mean": gq.mean()}).sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9), constrained_layout=True)
panels = [("e", "electric", "Electricity (all dwellings)"),
          ("g", "gas", "Gas (gas-heated dwellings)")]
for ax, (fc, sf, lbl) in zip(axes, panels):
    env = serl_envelope(sf); h = env.index.to_numpy()
    ax.fill_between(h, env["lo"], env["hi"], color=SERL_GREY, alpha=0.20, label="SERL 10\u201390%")
    ax.plot(h, env["mean"], color=SERL_GREY, lw=1.4, ls="--", label="SERL mean")
    ax.plot(PROF[fc]["winter"].index, PROF[fc]["winter"].values, color=WINTER_C, lw=2.2,
            label="modelled winter")
    ax.plot(PROF[fc]["summer"].index, PROF[fc]["summer"].values, color=SUMMER_C, lw=2.2,
            label="modelled summer")
    ax.axhline(1.0, color=SERL_GREY, lw=0.6, ls=":")
    r = np.corrcoef(PROF[fc]["winter"].reindex(h).values, env["mean"].values)[0, 1]
    ax.set_title(f"{lbl}   (winter shape corr {r:.2f})", fontsize=10.5)
    ax.set_xlabel("hour of day (local)"); ax.set_xticks(range(0, 24, 4))
    ax.grid(True, color="#eeeeee", zorder=0)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.legend(fontsize=7.5, ncol=2, frameon=False, loc="upper left")
axes[0].set_ylabel("relative demand (mean = 1)")

out = FIGDIR / "figure_intraday_shape.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"wrote {out}")

## Figure 4 — `figure_diurnal_by_heating_type.png`
The config's off-peak heating charge profile vs SERL storage-radiator and
direct-electric diurnal curves (new; reads `L.DIURNAL`).

In [ ]:
def diurnal_cht(val):   # SERL electricity hourly for one central_heating_type
    s = diurnal[(diurnal.quantity=='Electricity imports')&(diurnal.seg3_var=='central_heating_type')&
                (diurnal.seg3_value==val)&(diurnal.year==L.YEAR)&(diurnal.weekday_weekend==L.WKND)&(diurnal.has_pv==L.PV)]
    v = s.groupby('hour')['mean_kwh'].apply(lambda x: pd.to_numeric(x,errors='coerce').mean()).reindex(range(24)).to_numpy(float)
    return v/np.nanmean(v)

hpo = L.heating_diurnal_storage()          # 24h mean-1.0 off-peak charge profile (the config value)
fig, ax = plt.subplots(figsize=(7.8, 3.2))
ax.plot(range(24), hpo, 's-', color='#1d8a55', label='off-peak heating profile (config)')
ax.plot(range(24), diurnal_cht('Electric storage radiators'), 'o--', color='#2c6fbb', alpha=.6, label='SERL storage radiators (total elec)')
ax.plot(range(24), diurnal_cht('Electric radiators'), '^:', color='#c0392b', alpha=.7, label='SERL direct electric (evening, ~gas)')
ax.axhline(1.0, color='#999', lw=.8, ls=':'); ax.set_xticks(range(0,24,3))
ax.set_xlabel('hour'); ax.set_ylabel('mean-1.0 multiplier')
ax.set_title('Off-peak electric heating charges overnight; direct electric peaks evening like gas')
ax.legend(fontsize=8); plt.tight_layout()

out = FIGDIR / "figure_diurnal_by_heating_type.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"heating_profile_24h_offpeak: peak hour {int(np.argmax(hpo))}, "
      f"overnight(0-6) {np.mean(hpo[:7]):.2f}x vs midday(9-17) {np.mean(hpo[9:18]):.2f}x")
print(f"wrote {out}")

## Figure 5 — `figure_desnz_validation.png`
Inter-annual (Newcastle 2021-2023) and cross-sectional (five cities, 2023)
per-dwelling total energy, model vs DESNZ (ported from
`make_desnz_validation_figure.py`).

In [ ]:
MODEL_C, DESNZ_C = "#2E8B57", "#8a8a8a"
YEARS = [2021, 2022, 2023]
CITIES5 = [
    ("newcastle", "newcastle", "Newcastle", "#2c5f9e"),
    ("sunderland", "sunderland", "Sunderland", "#e1a100"),
    ("waltham_forest", "waltham_forest", "Waltham Forest", "#8c564b"),
    ("manchester", "manchester", "Manchester", "#c0392b"),
    ("brighton", "brighton_and_hove", "Brighton & Hove", "#7d5bbe"),
]

def per_dwelling(roll, des):
    mm = roll.merge(des, on="lsoa_code", how="inner")
    mm["t_model"] = (mm.abm_elec_kwh + mm.abm_gas_kwh) / mm.run_dwellings / 1000.0
    mm["t_desnz"] = (mm.total_kwh_elec + mm.total_kwh_gas) / mm.meters_elec / 1000.0
    return mm.replace([np.inf, -np.inf], np.nan).dropna(subset=["t_model", "t_desnz"])

model_yr, desnz_yr = [], []
for yr in YEARS:
    mm = per_dwelling(
        pd.read_csv(REPO / f"results_lsoa/transfer_v2_newcastle/abm_year_all_newcastle_{yr}.csv"),
        load_desnz("newcastle", yr))
    model_yr.append((mm.t_model * mm.run_dwellings).sum() / mm.run_dwellings.sum())
    desnz_yr.append((mm.t_desnz * mm.meters_elec).sum() / mm.meters_elec.sum())

frames = []
for desnz_slug, dirslug, name, colour in CITIES5:
    mm = per_dwelling(
        pd.read_csv(REPO / f"results_lsoa/transfer_v2_{dirslug}/abm_year_all_{dirslug}_2023.csv"),
        load_desnz(desnz_slug, 2023))
    mm["city"] = name; mm["colour"] = colour
    frames.append(mm[["city", "colour", "t_model", "t_desnz"]])
allc = pd.concat(frames, ignore_index=True)
r = np.corrcoef(allc.t_desnz, allc.t_model)[0, 1]
mape = (np.abs(allc.t_model - allc.t_desnz) / allc.t_desnz).mean() * 100
bias = ((allc.t_model - allc.t_desnz) / allc.t_desnz).mean() * 100

fig, (axA, axB) = plt.subplots(1, 2, figsize=(12.6, 5.3), constrained_layout=True)
x = np.arange(len(YEARS)); w = 0.38
bA = axA.bar(x - w / 2, model_yr, w, color=MODEL_C, label="model")
bD = axA.bar(x + w / 2, desnz_yr, w, color=DESNZ_C, label="DESNZ metered")
for bars in (bA, bD):
    axA.bar_label(bars, fmt="%.1f", fontsize=8, padding=2, color="#555")
axA.set_xticks(x); axA.set_xticklabels([str(y) for y in YEARS])
axA.set_ylabel("total energy (MWh / dwelling / year)")
axA.set_ylim(0, max(desnz_yr) * 1.18)
axA.legend(frameon=False, fontsize=9, loc="upper right")
axA.set_title("Newcastle, 2021\u20132023", fontsize=10.5)
axA.grid(axis="y", color="#eeeeee"); axA.set_axisbelow(True)

lim = max(allc.t_desnz.max(), allc.t_model.max()) * 1.05
axB.plot([0, lim], [0, lim], color="#999999", lw=1, ls="--", zorder=1, label="1:1")
for name, colour in [(c[2], c[3]) for c in CITIES5]:
    dsub = allc[allc.city == name]
    axB.scatter(dsub.t_desnz, dsub.t_model, s=14, alpha=0.6, color=colour, edgecolor="none",
                zorder=3, label=name)
axB.set_xlim(0, lim); axB.set_ylim(0, lim); axB.set_aspect("equal")
axB.set_xlabel("DESNZ metered (MWh / dwelling / year)")
axB.set_ylabel("modelled (MWh / dwelling / year)")
axB.text(0.04, 0.96, f"pooled r = {r:.2f}\nMAPE = {mape:.0f} %\nmean bias = {bias:+.0f} %",
         transform=axB.transAxes, va="top", ha="left", fontsize=9,
         bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#dddddd"))
axB.legend(loc="lower right", frameon=False, fontsize=8)
axB.set_title("Five cities, per LSOA (2023)", fontsize=10.5)
axB.grid(True, color="#f0f0f0")
for ax in (axA, axB):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

out = FIGDIR / "figure_desnz_validation.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"wrote {out}")
print(f"  (a) model {[round(v,1) for v in model_yr]}  DESNZ {[round(v,1) for v in desnz_yr]}")
print(f"  (b) n={len(allc)}  r={r:.2f}  MAPE={mape:.1f}%  bias={bias:+.1f}%")

## Figure 6 — `figure_coverage_scatter.png`
Five-city per-LSOA modelled-vs-DESNZ electricity (panel A) and ratio-vs-coverage
with the pooled fit (panel B), from the v2 `transfer_confidence` CSVs
(ported from `make_paper_figures.coverage_scatter`).

In [ ]:
from matplotlib.lines import Line2D

CONF_V2 = REPO / "research/applied/results/transfer_v2"
CITIESC = [
    ("newcastle", "Newcastle"),
    ("sunderland", "Sunderland"),
    ("waltham_forest", "Waltham Forest"),
    ("manchester", "Manchester"),
    ("brighton", "Brighton & Hove"),
]
CITY_COLOUR = {
    "Newcastle": "#0072B2", "Sunderland": "#E69F00", "Waltham Forest": "#009E73",
    "Manchester": "#CC79A7", "Brighton & Hove": "#D55E00",
}
TIER_COLOUR = {"High": "#2E8B57", "Medium": "#E1A100", "Low": "#C44E52"}
TIER_ORDER = ["High", "Medium", "Low"]

def _conf_csv(slug):
    return CONF_V2 / f"transfer_confidence_{slug}_2023_v2.csv"

def _load_all_cities():
    frames = []
    for slug, name in CITIESC:
        d = pd.read_csv(_conf_csv(slug)); d["city"] = name; frames.append(d)
    return pd.concat(frames, ignore_index=True)

def _ols(y, x):
    X = np.column_stack([np.ones(len(x)), x])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ b
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return b, r2

d = _load_all_cities()
fig, (axA, axB) = plt.subplots(1, 2, figsize=(12.5, 5.6), constrained_layout=True)

lo, hi = np.inf, 0.0
for _, name in CITIESC:
    s = d[d["city"] == name]
    x = s["total_kwh_elec"].to_numpy() / 1e6
    y = s["abm_elec_kwh"].to_numpy() / 1e6
    ok = (x > 0) & (y > 0)
    axA.scatter(x[ok], y[ok], s=11, alpha=0.6, color=CITY_COLOUR[name], edgecolor="none")
    lo = min(lo, np.nanmin(x[ok]), np.nanmin(y[ok]))
    hi = max(hi, np.nanmax(x[ok]), np.nanmax(y[ok]))
lo, hi = lo * 0.8, hi * 1.25
axA.plot([lo, hi], [lo, hi], color="grey", lw=1.0, ls="--")
axA.set_xscale("log"); axA.set_yscale("log")
axA.set_xlim(lo, hi); axA.set_ylim(lo, hi)
axA.set_xlabel("DESNZ metered electricity (GWh per LSOA)", fontsize=9)
axA.set_ylabel("Modelled electricity (GWh per LSOA)", fontsize=9)
axA.set_title("A", fontsize=11, loc="left", fontweight="bold")
axA.tick_params(labelsize=8)

v = d[["coverage", "tot_ratio", "city"]].replace([np.inf, -np.inf], np.nan).dropna()
for _, name in CITIESC:
    s = v[v["city"] == name]
    axB.scatter(s["coverage"], s["tot_ratio"], s=11, alpha=0.6,
                color=CITY_COLOUR[name], edgecolor="none")
b, r2 = _ols(v["tot_ratio"].to_numpy(), v["coverage"].to_numpy())
xs = np.linspace(v["coverage"].min(), v["coverage"].max(), 50)
axB.plot(xs, b[0] + b[1] * xs, color="black", lw=1.2)
axB.axhline(1.0, color="grey", lw=0.8, ls="--")
axB.text(0.04, 0.95, f"pooled R\u00b2 = {r2:.2f}", transform=axB.transAxes, fontsize=8.5, va="top")
axB.set_xlabel("Coverage (modelled dwellings / DESNZ meters)", fontsize=9)
axB.set_ylabel("Ratio (modelled / metered electricity)", fontsize=9)
axB.set_title("B", fontsize=11, loc="left", fontweight="bold")
axB.tick_params(labelsize=8)

handles = [Line2D([0], [0], marker="o", ls="", color=CITY_COLOUR[n], markersize=6, label=n)
           for _, n in CITIESC]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9,
           frameon=False, bbox_to_anchor=(0.5, -0.04))

out = FIGDIR / "figure_coverage_scatter.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"wrote {out}")

## Figure 7 — `figure_confidence_map.png`
Newcastle LSOA reliability-tier choropleth (ported from
`make_paper_figures.confidence_map`). The contextily basemap needs network; if
tiles cannot be fetched the choropleth is still saved without a basemap.

In [ ]:
BOUNDARIES = REPO / "data/boundaries/newcastle_lsoa_2021.geojson"

bound = gpd.read_file(BOUNDARIES)
conf = pd.read_csv(_conf_csv("newcastle"))[["lsoa_code", "confidence"]]
g = bound.merge(conf, left_on="LSOA21CD", right_on="lsoa_code", how="inner")
g = g.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(8, 8.4), constrained_layout=True)
for tier in TIER_ORDER:
    sub = g[g["confidence"] == tier]
    if not sub.empty:
        sub.plot(ax=ax, color=TIER_COLOUR[tier], edgecolor="white",
                 linewidth=0.4, alpha=0.72)
basemap_ok = False
try:
    import contextily as cx
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, attribution_size=5)
    basemap_ok = True
except Exception as e:  # offline / missing: keep the choropleth, drop tiles
    print(f"  basemap tiles unavailable ({e}); rendering without basemap")
ax.set_axis_off()
counts = g["confidence"].value_counts()
handles = [plt.Rectangle((0, 0), 1, 1, fc=TIER_COLOUR[t], ec="white",
                         alpha=0.72, label=f"{t}  (n={int(counts.get(t, 0))})")
           for t in TIER_ORDER]
ax.legend(handles=handles, loc="upper left", fontsize=10, frameon=True,
          title="Reliability tier", title_fontsize=10)

out = FIGDIR / "figure_confidence_map.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"wrote {out} ({len(g)} LSOAs, basemap={'yes' if basemap_ok else 'NO'})")

## Figure 8 — `figure_dimension_reproduction.png`
Assembled model output vs SERL, one calibrated dimension at a time, from the
shared Newcastle segmentation run (new; uses `seg`, `daily`, `cell`).

In [ ]:
DAYS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
ORD = {"floor_area_m2":       ["50 or less", "51 to 100", "101 to 150", "151 to 200", "Over 200"],
       "currentEnergyRating": ["A and B", "C", "D", "E", "F and G"],
       "num_occupants":       ["1", "2", "3", "4", "5", ">=6"]}

def _model_annual_by(segn):
    d = seg[seg.segmentation == segn].copy()
    d["ts"] = pd.to_datetime(d.timestamp_utc, utc=True)
    g = d.groupby(["value", "ts"], as_index=False).agg(
        e=("electric_kwh", "sum"), gg=("gas_kwh", "sum"), n=("n_homes", "sum"))
    g = g[g.n > 0].copy()
    g["eph"] = g.e / g.n; g["gph"] = g.gg / g.n
    return g.groupby("value").agg(elec=("eph", "sum"), gas=("gph", "sum"))

def _serl_annual_by(q, segn):
    out = {}
    for v in daily[daily.seg3_var == segn].seg3_value.dropna().unique().astype(str):
        m = pd.to_numeric(cell(q, "All", segn, v).set_index("month")["mean"],
                          errors="coerce").reindex(range(1, 13)).values
        if np.isfinite(m).sum() >= 10:
            out[v] = float(np.nansum(m * DAYS))
    return pd.Series(out)

DIMS = [("floor_area_m2", "floor area"), ("currentEnergyRating", "EPC band"),
        ("building_type", "property type"), ("num_occupants", "occupants")]
rows = []
fig, axes = plt.subplots(2, len(DIMS), figsize=(3.6 * len(DIMS), 6.6))
for j, (segn, label) in enumerate(DIMS):
    ma = _model_annual_by(segn)
    for rr, (q, col, ylab) in enumerate([("Electricity imports", "elec", "electricity"),
                                         ("Gas", "gas", "gas")]):
        sa = _serl_annual_by(q, segn)
        cats = [c for c in (ORD.get(segn) or list(sa.index)) if c in sa.index and c in ma.index]
        ax = axes[rr][j]
        if len(cats) >= 2:
            mv = ma.loc[cats, col].to_numpy(float); sv = sa.loc[cats].to_numpy(float)
            x = np.arange(len(cats))
            ax.plot(x, sv, "o-", color="#c0392b", label="SERL")
            ax.plot(x, mv, "s--", color="#1d8a55", label="ABM combined")
            ax.set_xticks(x); ax.set_xticklabels(cats, rotation=40, ha="right", fontsize=7)
            rows.append(dict(dimension=label, fuel=ylab, categories=len(cats),
                             shape_r=round(float(np.corrcoef(mv, sv)[0, 1]), 3) if len(cats) > 2 else np.nan,
                             level_gap=f"{100*(mv.sum()-sv.sum())/sv.sum():+.0f}%"))
        else:
            ax.text(0.5, 0.5, "no matched\ncategories", ha="center", va="center", transform=ax.transAxes, fontsize=8)
        if rr == 0: ax.set_title(label, fontsize=10)
        if j == 0: ax.set_ylabel(f"{ylab}\nkWh/home/yr")
axes[0][0].legend(fontsize=8)
plt.suptitle("Assembled model vs SERL, one dimension at a time  "
             "(curves parallel = combination preserves the gradient)", y=1.02)
plt.tight_layout()

out = FIGDIR / "figure_dimension_reproduction.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print("shape_r = correlation of the model's across-category gradient with SERL's "
      "(near 1.0 = combination didn't distort the shape).")
print(f"wrote {out}")
pd.DataFrame(rows)